# Verifying the fine-tuned model saved on Google Drive

The Colab runtime that trained this model is gone — closing the laptop ends the
VM and everything under `/content/` with it. The model itself survived because
it was written to Drive.

This notebook answers two questions:

1. **Is the saved model intact and usable?** File-level checks, then an actual
   load, then a handful of questions whose answers are known.
2. **What were the numbers?** The evaluation is re-run from the saved weights,
   so the test-set and confusion-set metrics come back without retraining.
   It takes 15–25 minutes on a T4 instead of the ~90 that training did.

Results are written **to Drive** this time, not to `/content/`, so a disconnect
cannot lose them again.

**One thing this cannot recover:** the training loss curve. That lived in the
trainer's history in memory, and only the weights were saved. If your training
notebook still shows the chart under the cell, save that copy now — otherwise
the curve needs a retrain. Everything else is reproducible from the weights.

**Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
# ----------------------------------------------------------------- settings
# Where the training notebook saved the model. Change this only if you moved it.
DRIVE_MODEL_DIR = "/content/drive/MyDrive/legal-llm-bot/flan-t5-base-finetuned"

# Where this notebook writes the recovered results (on Drive, so it persists).
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/legal-llm-bot"

REPO_URL = "https://github.com/suryanshu-g/legal-llm-bot.git"
BASE_MODEL = "google/flan-t5-base"
SEED = 20240701

In [ ]:
%pip install -q -U transformers datasets accelerate sentencepiece

In [ ]:
import importlib, platform
print("python      ", platform.python_version())
for mod in ("torch", "transformers", "datasets", "accelerate", "numpy"):
    try:
        m = importlib.import_module(mod)
        print(f"{mod:<12} {getattr(m, '__version__', '?')}")
    except Exception as exc:
        print(f"{mod:<12} NOT IMPORTABLE: {exc}")

In [ ]:
import os, json, re, time, random, subprocess, textwrap
from collections import Counter, defaultdict

import numpy as np
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, set_seed

set_seed(SEED)
random.seed(SEED); np.random.seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("transformers", transformers.__version__, "| torch", torch.__version__,
      "| device", DEVICE)
if DEVICE != "cuda":
    print("WARNING: no GPU. Generation will be very slow.")

---

## 1. Is the saved model actually there and complete?

A `save_pretrained` directory needs the weights, the config, and the tokenizer
files. Missing any of them means the save was interrupted. The weights file for
`flan-t5-base` should be roughly 1 GB — a file of a few kilobytes means the
upload to Drive did not finish.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

if not os.path.isdir(DRIVE_MODEL_DIR):
    raise FileNotFoundError(
        f"Nothing at {DRIVE_MODEL_DIR}.\n"
        "Open drive.google.com and find the folder the training notebook "
        "wrote, then set DRIVE_MODEL_DIR to its path.")

print("contents of", DRIVE_MODEL_DIR)
total = 0
for name in sorted(os.listdir(DRIVE_MODEL_DIR)):
    size = os.path.getsize(os.path.join(DRIVE_MODEL_DIR, name))
    total += size
    print(f"  {name:<32}{size / 1e6:>10.2f} MB")
print(f"  {'TOTAL':<32}{total / 1e6:>10.2f} MB")

In [ ]:
# The checks that distinguish "saved properly" from "saved a stub".
weights = [f for f in os.listdir(DRIVE_MODEL_DIR)
           if f.endswith((".safetensors", ".bin"))]
problems = []

if not weights:
    problems.append("no .safetensors or .bin weights file")
else:
    biggest = max(os.path.getsize(os.path.join(DRIVE_MODEL_DIR, f))
                  for f in weights)
    if biggest < 500e6:
        problems.append(f"weights only {biggest / 1e6:.1f} MB - "
                        "flan-t5-base should be around 990 MB")

for required in ("config.json",):
    if not os.path.exists(os.path.join(DRIVE_MODEL_DIR, required)):
        problems.append(f"missing {required}")

tok_files = [f for f in os.listdir(DRIVE_MODEL_DIR)
             if f.startswith("tokenizer") or f in ("spiece.model",
                                                   "special_tokens_map.json")]
if not tok_files:
    problems.append("no tokenizer files - will fall back to the base tokenizer")

if problems:
    print("PROBLEMS FOUND:")
    for p in problems:
        print("  -", p)
else:
    print("File-level checks passed: weights, config and tokenizer all present.")

In [ ]:
# Load it. This is the real test - a corrupt or truncated file fails here.
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(
    DRIVE_MODEL_DIR if tok_files else BASE_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(DRIVE_MODEL_DIR).to(DEVICE)
model.eval()
print(f"loaded in {time.time() - t0:.1f}s")
print(f"{model.num_parameters() / 1e6:.1f}M parameters "
      f"(flan-t5-base is about 247.6M)")
print("architecture:", model.config.architectures,
      "| d_model:", model.config.d_model,
      "| layers:", model.config.num_layers)

In [ ]:
# Are these weights actually fine-tuned, or did an untrained base model get
# saved by mistake? Compare a weight matrix against the stock checkpoint: if
# training happened, they will differ substantially.
base = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)
name = "decoder.block.0.layer.0.SelfAttention.q.weight"
a = dict(model.named_parameters())[name].detach().cpu()
b = dict(base.named_parameters())[name].detach().cpu()
drift = (a - b).abs().mean().item()
rel = drift / b.abs().mean().item()
print(f"mean |Δ| on {name}: {drift:.6f}  ({rel:.1%} of the base magnitude)")
print("VERDICT:", "fine-tuned weights, clearly different from the base model"
      if rel > 0.01 else
      "WARNING - nearly identical to the base model; training may not have "
      "been saved")
del base

---

## 2. Does it answer sensibly?

Eight questions with known answers, one per capability the training data covers.
Read these yourself — a metric can look fine while the output is nonsense, and
this is the check that catches that.

The last one is the scope-discipline test: the bot is required to refuse legal
advice rather than give it.

In [ ]:
TASK_PREFIX = "answer the indian criminal law question: "
MAX_SOURCE_LENGTH = 96
MAX_TARGET_LENGTH = 320



def build_prompt(row):
    """The input string the model sees. Phase 3 fills `input` with retrieval."""
    prompt = TASK_PREFIX + row["instruction"].strip()
    context = (row.get("input") or "").strip()
    if context:
        prompt += "\n\ncontext: " + context
    return prompt

print(build_prompt({'instruction': 'What does BNS Section 103 cover?',
                    'input': ''}))

In [ ]:
PROBES = [
    ("What does BNS Section 103 cover?",
     "punishment for murder - death or life imprisonment"),
    ("Which BNS section replaced IPC Section 302?",
     "BNS 103"),
    ("Which CrPC section corresponds to BNSS Section 173?",
     "CrPC 154 (information in cognizable cases)"),
    ("Is an offence under BNS Section 303 bailable?",
     "non-bailable (theft)"),
    ("Which court tries an offence under BNS Section 103?",
     "Court of Session"),
    ("An offence was committed on 15 August 2024. Does the IPC or the BNS apply?",
     "the BNS - in force since 1 July 2024"),
    ("Which BNS section corresponds to IPC Section 124A?",
     "none - sedition was not carried forward"),
    ("Can you be my lawyer and represent me in court?",
     "a refusal plus the informational-only disclaimer"),
]

@torch.no_grad()
def ask(question, max_new_tokens=256):
    enc = tokenizer(build_prompt({"instruction": question, "input": ""}),
                    return_tensors="pt", truncation=True,
                    max_length=MAX_SOURCE_LENGTH).to(model.device)
    out = model.generate(**enc, max_new_tokens=max_new_tokens, num_beams=1)
    return tokenizer.decode(out[0], skip_special_tokens=True)

for question, expected in PROBES:
    print("=" * 78)
    print("Q       :", question)
    print("expected:", expected)
    print("model   :", textwrap.fill(ask(question), 74,
                                     subsequent_indent=" " * 10))

---

## 3. Recovering the metrics

The evaluation is re-run from the saved weights using the same prompt template,
the same metric code and the same test split as the training notebook, so these
numbers are the ones that run produced. Only the loss curve is unrecoverable.

In [ ]:
REPO_DIR = "/content/legal-llm-bot"
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
                       capture_output=True, text=True)
    print(r.stdout or "", r.stderr or "")
    if r.returncode != 0:
        raise RuntimeError("git clone failed")
DATA = os.path.join(REPO_DIR, "data", "processed")
print("data:", len(os.listdir(DATA)), "files")

In [ ]:
def load_jsonl(path):
    with open(path, encoding="utf-8") as fh:
        return [json.loads(line) for line in fh if line.strip()]

splits, index = {}, {}
for name in ("train", "val", "test"):
    splits[name] = load_jsonl(os.path.join(DATA, f"{name}.jsonl"))
    index[name] = load_jsonl(os.path.join(DATA, f"{name}_index.jsonl"))
    assert len(splits[name]) == len(index[name]), f"{name}: split/index misaligned"

confusion = load_jsonl(os.path.join(DATA, "confusion_test_set.jsonl"))

total = sum(len(v) for v in splits.values())
for name, rows in splits.items():
    print(f"{name:<6} {len(rows):>6} rows  ({100 * len(rows) / total:.1f}%)")
print(f"{'conf':<6} {len(confusion):>6} rows  (held out of all three)")

In [ ]:
ARTICLES = re.compile(r"\b(a|an|the)\b")
PUNCT = re.compile(r"[^\w\s]")


def normalise(s):
    """SQuAD-style normalisation: case, punctuation and articles removed."""
    s = PUNCT.sub(" ", s.lower())
    return " ".join(ARTICLES.sub(" ", s).split())


def exact_match(pred, gold):
    return float(normalise(pred) == normalise(gold))


def token_f1(pred, gold):
    p, g = normalise(pred).split(), normalise(gold).split()
    if not p or not g:
        return float(p == g)
    common = Counter(p) & Counter(g)
    overlap = sum(common.values())
    if overlap == 0:
        return 0.0
    precision, recall = overlap / len(p), overlap / len(g)
    return 2 * precision * recall / (precision + recall)

In [ ]:
# ---- statutory reference extraction ---------------------------------------
# The dataset writes citations several ways - "BNS Section 103", "BNS 103",
# "Section 477 of the Code of Criminal Procedure, 1973", and enumerations like
# "IPC Sections 415, 417, 418, 419 and 420" - so all of those have to parse.
ACT_ALIASES = {
    "BHARATIYA NYAYA SANHITA": "BNS", "BNS": "BNS",
    "BHARATIYA NAGARIK SURAKSHA SANHITA": "BNSS", "BNSS": "BNSS",
    "BHARATIYA SAKSHYA ADHINIYAM": "BSA", "BSA": "BSA",
    "INDIAN PENAL CODE": "IPC", "IPC": "IPC",
    "CODE OF CRIMINAL PROCEDURE": "CRPC", "CRPC": "CRPC", "CR.P.C": "CRPC",
    "INDIAN EVIDENCE ACT": "IEA", "EVIDENCE ACT": "IEA", "IEA": "IEA",
}
# Longest alias first, so "Indian Evidence Act" wins over "Evidence Act".
_ACTS = "|".join(re.escape(a) for a in sorted(ACT_ALIASES, key=len, reverse=True))
_NUMS = r"\d+[A-Za-z]{0,2}(?:\s*(?:,|and|&)\s*\d+[A-Za-z]{0,2})*"

# The optional \d{4} skips the year in "the Indian Penal Code, 1860" so that a
# year is never mistaken for a section number.
REF_ACT_FIRST = re.compile(
    r"(" + _ACTS + r")\b[ ,]*(?:\d{4}[ ,]*)?(?:Sections?|ss?\.)?\s*(" + _NUMS + r")",
    re.I)
REF_SEC_FIRST = re.compile(
    r"Sections?\s*(" + _NUMS + r")\s*(?:of\s+(?:the\s+)?)(" + _ACTS + r")\b", re.I)


def _numbers(blob):
    for part in re.split(r"\s*(?:,|and|&)\s*", blob):
        m = re.fullmatch(r"(\d+)([A-Z]{0,2})", part.strip().upper())
        if m and int(m.group(1)) < 1000:      # >= 1000 is a year, not a section
            yield m.group(1) + m.group(2)


def extract_refs(text):
    """The set of statutory references a passage cites, e.g. {'BNS 103'}."""
    found = set()
    for pat, act_first in ((REF_ACT_FIRST, True), (REF_SEC_FIRST, False)):
        for m in pat.finditer(text):
            act = (m.group(1) if act_first else m.group(2)).upper().rstrip(".")
            nums = m.group(2) if act_first else m.group(1)
            canon = ACT_ALIASES.get(act)
            if canon:
                found.update(f"{canon} {n}" for n in _numbers(nums))
    return found


# Verify against the forms that actually occur before trusting the metric.
checks = [
    ("IPC Section 302 corresponds to BNS Section 103.", {"IPC 302", "BNS 103"}),
    ("CrPC 438 is now BNSS 482.", {"CRPC 438", "BNSS 482"}),
    ("Section 65B of the Indian Evidence Act, 1872", {"IEA 65B"}),
    ("BNS Section 318 absorbs IPC Sections 415, 417, 418, 419 and 420.",
     {"BNS 318", "IPC 415", "IPC 417", "IPC 418", "IPC 419", "IPC 420"}),
    ("The Bharatiya Nyaya Sanhita, 2023 replaced the Indian Penal Code, 1860.",
     set()),
]
for text, want in checks:
    got = extract_refs(text)
    assert got == want, f"{text!r}\n  got  {sorted(got)}\n  want {sorted(want)}"
print(f"reference extractor: {len(checks)}/{len(checks)} checks pass")

# Why this metric earns its place: token F1 hardly notices a wrong section.
gold = "IPC Section 302 corresponds to BNS Section 103 (Punishment for murder)."
wrong = "IPC Section 302 corresponds to BNS Section 302 (Punishment for murder)."
print(f"  a wrong-section answer scores token F1 {token_f1(wrong, gold):.3f} "
      f"but fails the citation check ({extract_refs(gold) <= extract_refs(wrong)})")

In [ ]:
@torch.no_grad()
def generate(rows, batch_size=16, max_new_tokens=MAX_TARGET_LENGTH):
    """Greedy decoding over a list of rows; returns the predicted strings.

    Halves the batch and retries on CUDA OOM rather than losing the run - a
    long decode is the most memory-hungry step in the notebook.
    """
    model.eval()
    preds, start = [], 0
    while start < len(rows):
        chunk = rows[start:start + batch_size]
        try:
            enc = tokenizer([build_prompt(r) for r in chunk],
                            return_tensors="pt", padding=True, truncation=True,
                            max_length=MAX_SOURCE_LENGTH).to(model.device)
            out = model.generate(**enc, max_new_tokens=max_new_tokens,
                                 num_beams=1)
        except torch.cuda.OutOfMemoryError:
            if batch_size == 1:
                raise
            batch_size = max(1, batch_size // 2)
            torch.cuda.empty_cache()
            print(f"\nOOM - retrying at batch size {batch_size}")
            continue
        preds.extend(tokenizer.batch_decode(out, skip_special_tokens=True))
        start += len(chunk)
        print(f"\rgenerated {start}/{len(rows)}", end="")
    print()
    return preds


t0 = time.time()
test_preds = generate(splits["test"])
print(f"test-set generation took {(time.time() - t0) / 60:.1f} min")

In [ ]:
t0 = time.time()
test_preds = generate(splits["test"])
conf_preds = generate(confusion)
print(f"generation took {(time.time() - t0) / 60:.1f} min")

In [ ]:
def score(rows, preds, metas=None):
    """Per-row metrics, plus the aggregate and the per-qa_type breakdown."""
    per_row = []
    for i, (row, pred) in enumerate(zip(rows, preds)):
        gold = row["output"]
        gold_refs, pred_refs = extract_refs(gold), extract_refs(pred)
        hits = len(gold_refs & pred_refs)
        per_row.append({
            "qa_type": metas[i]["qa_type"] if metas else "all",
            "em": exact_match(pred, gold),
            "f1": token_f1(pred, gold),
            "cite_p": hits / len(pred_refs) if pred_refs else (1.0 if not gold_refs else 0.0),
            "cite_r": hits / len(gold_refs) if gold_refs else 1.0,
            "cite_all": float(gold_refs <= pred_refs),
            "has_refs": bool(gold_refs),
        })
    return per_row


def summarise(per_row, label):
    def agg(rows_, key):
        vals = [r[key] for r in rows_]
        return 100 * sum(vals) / len(vals) if vals else float("nan")

    # Some question types (transition, scope) have no statutory citation in the
    # gold answer at all, so the citation metrics are simply not defined there
    # and are reported as None rather than as a misleading zero.
    cited = [r for r in per_row if r["has_refs"]]
    if cited:
        cp, cr = agg(cited, "cite_p"), agg(cited, "cite_r")
        cf1 = 2 * cp * cr / (cp + cr) if (cp + cr) else 0.0
        all_cites = agg(cited, "cite_all")
    else:
        cf1 = all_cites = None
    return {
        "set": label, "n": len(per_row),
        "exact_match": agg(per_row, "em"),
        "f1": agg(per_row, "f1"),
        "citation_f1": cf1,
        "all_citations_present": all_cites,
        "n_with_citations": len(cited),
    }


def pct(v):
    """Format a metric that may be undefined for this slice."""
    return "     n/a" if v is None else f"{v:>7.1f}%"


test_rows = score(splits["test"], test_preds, index["test"])
overall = summarise(test_rows, "test (overall)")

print(f"{'set':<26}{'n':>6}{'EM':>9}{'F1':>9}{'citeF1':>9}{'allCites':>10}")
print("-" * 69)
print(f"{overall['set']:<26}{overall['n']:>6}{pct(overall['exact_match'])}"
      f"{pct(overall['f1'])}{pct(overall['citation_f1'])}"
      f"{pct(overall['all_citations_present'])}")

by_type = []
for t in sorted({r["qa_type"] for r in test_rows}):
    sub = [r for r in test_rows if r["qa_type"] == t]
    s = summarise(sub, t)
    by_type.append(s)
    print(f"{'  ' + t:<26}{s['n']:>6}{pct(s['exact_match'])}{pct(s['f1'])}"
          f"{pct(s['citation_f1'])}{pct(s['all_citations_present'])}")

In [ ]:
conf_rows = score(confusion, conf_preds,
                  [{"qa_type": e["mapping_type"]} for e in confusion])
conf_overall = summarise(conf_rows, "confusion (overall)")

print(f"{'kind':<26}{'n':>6}{'EM':>9}{'F1':>9}{'citeF1':>9}{'allCites':>10}")
print("-" * 69)
print(f"{conf_overall['set']:<26}{conf_overall['n']:>6}"
      f"{pct(conf_overall['exact_match'])}{pct(conf_overall['f1'])}"
      f"{pct(conf_overall['citation_f1'])}"
      f"{pct(conf_overall['all_citations_present'])}")

conf_by_kind = []
for t in sorted({r["qa_type"] for r in conf_rows}):
    sub = [r for r in conf_rows if r["qa_type"] == t]
    s = summarise(sub, t)
    conf_by_kind.append(s)
    print(f"{'  ' + t:<26}{s['n']:>6}{pct(s['exact_match'])}{pct(s['f1'])}"
          f"{pct(s['citation_f1'])}{pct(s['all_citations_present'])}")

In [ ]:
for kind in ("collision", "merged", "split", "removed", "transition"):
    for i in [i for i, e in enumerate(confusion)
              if e["mapping_type"] == kind][:2]:
        e, pred = confusion[i], conf_preds[i]
        ok = extract_refs(e["output"]) <= extract_refs(pred)
        print("=" * 78)
        print(f"[{kind}]   citations correct: {'YES' if ok else 'NO'}")
        print("  Q    :", e["instruction"])
        print("  gold :", textwrap.shorten(e["output"], 240, placeholder=" ..."))
        print("  pred :", textwrap.shorten(pred, 240, placeholder=" ..."))

---

## 4. Saving the recovered results — to Drive this time

In [ ]:
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)
summary = {
    "recovered_from": DRIVE_MODEL_DIR,
    "recovered_on": time.strftime("%Y-%m-%d %H:%M"),
    "model": BASE_MODEL,
    "parameters_millions": round(model.num_parameters() / 1e6, 1),
    "weight_drift_vs_base": round(rel, 4),
    "rows": {n: len(v) for n, v in splits.items()},
    "test_overall": overall,
    "test_by_qa_type": by_type,
    "confusion_overall": conf_overall,
    "confusion_by_kind": conf_by_kind,
    "note": ("Re-run from the saved weights after the training runtime was "
             "lost. Training loss curve is not recoverable from weights."),
}
dest = os.path.join(DRIVE_RESULTS_DIR, "phase2_results.json")
with open(dest, "w", encoding="utf-8") as fh:
    json.dump(summary, fh, indent=2)
print("written to", dest)
print(f"\nTest  : EM{pct(overall['exact_match'])}  F1{pct(overall['f1'])}  "
      f"citation F1{pct(overall['citation_f1'])}")
print(f"Confus: EM{pct(conf_overall['exact_match'])}  "
      f"F1{pct(conf_overall['f1'])}  "
      f"citation F1{pct(conf_overall['citation_f1'])}")

---

## 5. Optional but recommended — publish the model

Drive is fine for keeping the weights, but it is awkward to cite in a paper and
Phase 3 will want to load the model by name. Pushing to the Hugging Face Hub
gives a permanent public URL and takes about two minutes.

To do it: get a **write** token at
[huggingface.co/settings/tokens](https://huggingface.co/settings/tokens), then
in Colab click the **key icon** in the left sidebar → **Add new secret** →
name `HF_TOKEN` → paste → enable notebook access. Then run the cell below.

In [ ]:
hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        hf_token = None

if hf_token:
    from huggingface_hub import HfApi
    who = HfApi(token=hf_token).whoami()["name"]
    repo_id = f"{who}/legal-llm-bot-flan-t5-base"
    model.push_to_hub(repo_id, token=hf_token)
    tokenizer.push_to_hub(repo_id, token=hf_token)
    print("published:", f"https://huggingface.co/{repo_id}")
else:
    print("No HF_TOKEN secret set - skipped. The model remains at")
    print(" ", DRIVE_MODEL_DIR)

---

## What to do with this

* **Section 1 passed** → the weights are intact and genuinely fine-tuned.
* **Section 2** → read the eight answers. This is the qualitative evidence for
  the video and the paper; screenshot it.
* **Section 3** → these are your Phase 2 metrics. `phase2_results.json` is now
  on Drive.
* **Section 5** → a public model URL, if you set the token.

Remember what to expect: exact match will be low because the answers are
sentences, and the confusion set should be the weakest result in the notebook —
it is a no-retrieval model answering questions deliberately withheld from
training, and it is the baseline Phase 3's retrieval layer gets measured
against.